[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Laverde97/phd-data-science-ai/blob/main/semesters/semester-01/machine-learning/notebooks/learning-guides/04-modelos-de-regresion.ipynb)

# Guía de aprendizaje 04 · Modelos de regresión

**Machine Learning · Semestre 1**

Esta guía convierte los conceptos de la presentación **Modelos de Regresión** en un recorrido práctico y fácil de estudiar.

## Objetivos de aprendizaje

Al finalizar deberías poder:

- explicar qué es un problema de regresión;
- distinguir variable predictora y variable objetivo;
- interpretar una regresión lineal simple;
- comprender qué representan intercepto, pendiente y residuo;
- entender la idea de mínimos cuadrados;
- separar entrenamiento y prueba;
- entrenar `LinearRegression` con scikit-learn;
- evaluar predicciones con métricas básicas;
- interpretar la ecuación final del modelo.


## 1. ¿Qué es regresión?

La **regresión** es una familia de técnicas de **aprendizaje supervisado** usada cuando la variable que queremos predecir es **numérica y continua**.

Ejemplos:
- salario;
- temperatura;
- precio;
- tiempo de entrega;
- ventas.

### Regla mental

> Si la respuesta esperada es un **número**, probablemente estamos ante un problema de regresión.

En cambio, si queremos predecir una categoría como “aprobado/no aprobado”, hablamos de **clasificación**.


## 2. Vocabulario esencial

| Concepto | Significado |
|---|---|
| **Variable predictora (`X`)** | Información utilizada para explicar o predecir. |
| **Variable objetivo (`y`)** | Número que queremos estimar. |
| **Modelo** | Función que aprende una relación entre `X` e `y`. |
| **Predicción (`ŷ`)** | Valor estimado por el modelo. |
| **Residuo** | Diferencia entre el valor real y la predicción. |
| **Entrenamiento** | Datos utilizados para ajustar el modelo. |
| **Prueba** | Datos reservados para evaluar generalización. |

En el ejemplo de clase:

- `X = experiencia`
- `y = salario`


## 3. Modelos de regresión mencionados en la presentación

La presentación introduce una caja de herramientas amplia:

1. **Regresión lineal simple**: una variable predictora.
2. **Regresión lineal múltiple**: varias variables predictoras.
3. **Regresión polinomial**: permite relaciones curvas.
4. **SVR**: Support Vector Regression.
5. **Árboles de decisión para regresión**.
6. **Bosques aleatorios para regresión**.

Esta guía implementa primero **regresión lineal simple**, porque permite comprender con claridad los fundamentos que luego se reutilizan en modelos más complejos.


## 4. Tres ideas que debe cumplir un buen modelo

### 4.1 Relación funcional
Buscamos una función matemática que aproxime la relación entre las variables.

### 4.2 Generalización
No basta con ajustar bien los datos conocidos. El verdadero objetivo es funcionar razonablemente bien con datos nuevos.

### 4.3 Interpretabilidad vs. potencia
Algunos modelos son muy fáciles de explicar; otros pueden capturar relaciones más complejas, pero son menos transparentes.

La elección del modelo depende del problema, los datos y el propósito del análisis.


## 5. Regresión lineal simple

La regresión lineal simple busca una recta:

\[
\hat{y} = b_0 + b_1x
\]

donde:

- \(\hat{y}\): predicción;
- \(b_0\): intercepto;
- \(b_1\): pendiente;
- \(x\): variable predictora.

Para el ejemplo:

\[
\widehat{salario} = b_0 + b_1(\text{experiencia})
\]

### ¿Cómo interpretar la pendiente?

Si `b1` es positiva, el salario estimado aumenta cuando aumenta la experiencia.  
Su magnitud indica cuánto cambia, en promedio, la predicción de salario por cada unidad adicional de experiencia.


## 6. ¿Cómo encuentra la “mejor” recta?

Cada observación tiene:

- un valor real \(y_i\);
- una predicción \(\hat{y}_i\).

La diferencia:

\[
e_i = y_i - \hat{y}_i
\]

se llama **residuo**.

El método de **mínimos cuadrados** busca los parámetros que minimizan:

\[
\sum_i (y_i - \hat{y}_i)^2
\]

Elevar al cuadrado evita que errores positivos y negativos se cancelen y penaliza con mayor fuerza los errores grandes.


## 7. Importar librerías

Usaremos:
- `pandas` para los datos;
- `matplotlib` para visualizar;
- `scikit-learn` para separar, entrenar y evaluar.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## 8. Cargar el conjunto salario–experiencia

La presentación trabaja con un conjunto donde cada fila relaciona años de experiencia y salario.

Usaremos la misma fuente CSV utilizada en clase.


In [ ]:
url = "https://cdtiueb.com.co/doctorado/salarios_experiencia_reg1.csv"
dataset = pd.read_csv(url)

dataset.head()


### Inspección rápida

Antes de modelar conviene revisar:
- dimensiones;
- nombres de columnas;
- tipos de datos;
- valores faltantes.


In [ ]:
print("Dimensiones:", dataset.shape)
print("\nColumnas:", list(dataset.columns))
print("\nTipos:")
print(dataset.dtypes)
print("\nValores faltantes:")
print(dataset.isna().sum())


## 9. Visualizar la relación

Un gráfico de dispersión ayuda a responder una pregunta esencial:

> ¿Parece existir una relación entre experiencia y salario?

Si los puntos siguen aproximadamente una tendencia recta, la regresión lineal puede ser un buen primer modelo.


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(dataset["experiencia"], dataset["salario"])
plt.xlabel("Años de experiencia")
plt.ylabel("Salario")
plt.title("Experiencia vs. salario")
plt.show()


## 10. Separar `X` e `y`

Scikit-learn espera `X` como una matriz de predictores y `y` como el vector objetivo.


In [ ]:
X = dataset[["experiencia"]].values
y = dataset["salario"].values

print("Forma de X:", X.shape)
print("Forma de y:", y.shape)


## 11. Dividir entrenamiento y prueba

Usaremos:
- 80 % para entrenamiento;
- 20 % para prueba;
- `random_state=0` para hacer reproducible la partición.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=0
)

print("Entrenamiento:", X_train.shape, y_train.shape)
print("Prueba:", X_test.shape, y_test.shape)


## 12. Entrenar el modelo

`LinearRegression().fit(X_train, y_train)` estima el intercepto y la pendiente de la recta usando los datos de entrenamiento.


In [ ]:
modelo = LinearRegression()
modelo.fit(X_train, y_train)

print("Intercepto:", modelo.intercept_)
print("Pendiente:", modelo.coef_[0])


## 13. Hacer predicciones

Ahora el modelo recibe valores de experiencia que no utilizó para ajustarse y estima sus salarios.


In [ ]:
y_pred = modelo.predict(X_test)

comparacion = pd.DataFrame({
    "experiencia": X_test.ravel(),
    "salario_real": y_test,
    "salario_predicho": y_pred,
    "residuo": y_test - y_pred
})

comparacion.head(10)


## 14. Visualizar la recta de regresión

Los puntos representan observaciones reales y la línea representa la predicción del modelo.


In [ ]:
orden = np.argsort(X_train.ravel())

plt.figure(figsize=(8, 5))
plt.scatter(X_train, y_train, label="Datos de entrenamiento")
plt.plot(
    X_train.ravel()[orden],
    modelo.predict(X_train)[orden],
    label="Recta de regresión"
)
plt.xlabel("Años de experiencia")
plt.ylabel("Salario")
plt.title("Regresión lineal simple")
plt.legend()
plt.show()


## 15. Evaluar el modelo

Estas métricas son una ampliación pedagógica para aprender a evaluar regresión:

- **MAE**: error absoluto promedio.
- **MSE**: promedio de errores al cuadrado.
- **RMSE**: raíz del MSE; vuelve a las unidades originales de `y`.
- **R²**: proporción de variabilidad explicada por el modelo.

### Regla general
Para MAE/MSE/RMSE, **más pequeño es mejor**.  
Para R², valores más cercanos a 1 indican mayor capacidad explicativa sobre esos datos.


In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MAE : {mae:,.2f}")
print(f"MSE : {mse:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R²  : {r2:.4f}")


## 16. Escribir la ecuación aprendida

La ecuación del modelo se construye con:

- `modelo.intercept_`
- `modelo.coef_[0]`


In [ ]:
b0 = modelo.intercept_
b1 = modelo.coef_[0]

print(f"salario_estimado = {b0:.2f} + ({b1:.2f} × experiencia)")


## 17. Predecir un nuevo caso

Podemos utilizar la ecuación aprendida para una persona con un número específico de años de experiencia.

Cambia el valor de `nueva_experiencia` y vuelve a ejecutar.


In [ ]:
nueva_experiencia = 5.0
prediccion = modelo.predict([[nueva_experiencia]])[0]

print(f"Experiencia: {nueva_experiencia} años")
print(f"Salario estimado: {prediccion:,.2f}")


## 18. ¿Qué hemos aprendido?

El flujo completo fue:

**datos → X/y → train/test → ajustar → predecir → evaluar → interpretar**

### Lo esencial

- La regresión predice valores numéricos.
- La regresión lineal simple usa una sola variable predictora.
- La pendiente describe el cambio esperado en `y` cuando cambia `x`.
- El intercepto representa el valor de la recta cuando `x = 0`.
- Los residuos miden los errores individuales.
- Mínimos cuadrados busca reducir la suma de residuos al cuadrado.
- El conjunto de prueba permite evaluar generalización.

## 19. Preguntas de autoevaluación

1. ¿Cuál es la diferencia entre clasificación y regresión?
2. ¿Qué representa `b1`?
3. ¿Qué es un residuo?
4. ¿Por qué no deberíamos evaluar únicamente sobre los datos de entrenamiento?
5. ¿Qué significa que R² sea alto?
6. ¿En qué se diferencia regresión lineal simple de múltiple?
7. ¿Cuándo una línea recta podría ser insuficiente?

## 20. Reto opcional

Prueba una de estas extensiones:

- cambia `test_size`;
- cambia `random_state`;
- compara predicción vs. realidad con un gráfico;
- calcula los residuos y analiza dónde son mayores;
- investiga cuándo sería apropiada una regresión polinomial.

> Esta guía implementa regresión lineal simple. Los demás modelos mencionados en la presentación (múltiple, polinomial, SVR, árboles y bosques) pueden desarrollarse como guías posteriores.
